# M9c · GBLA paper extras (prior architecture only)

**Scope.** Everything here is for the **previous GBLA / LN-tuned CLIP** experiment (M1–M9b): frozen CLIP ViT-L/14 + 50 LayerNorms + linear head, **104,450** trainable numbers. Nothing from M10/M11/M12 (SB-LN) is trained or claimed here.

**Five priorities (fixed before running):**

| # | What | GPU? | About |
|---|---|---|---|
| 1 | Confusion matrices, precision / recall / F1 / specificity, UADFV ROC from **saved** prediction files | no | seconds–minutes |
| 2 | Celeb-DF v2 test of the **9** existing checkpoints | eval only | ~2 min/model after crops are loaded |
| 3 | Seeds **1** and **2** for pooled (λ=∞) and GBLA (λ=0.01); seed 0 already in M4 | train | ~1.3 h × 4 ≈ 5–8 h |
| 4 | Training-order check: **interleaved** pure-generator batches instead of one long block per generator (seed 0, λ=∞ and λ=0.01) | train | ~1.3 h × 2 ≈ 3 h |
| 5 | Inference latency, FLOPs, peak memory | yes, short | minutes |

**Models (same 9 as M8b/M9b).** `m1`, `lam_inf`, `lam_1`, `lam_0.1`, **`lam_0.01`**, `lam_0.001`, `lam_0`, `uniform`, `fisher`.

**Celeb-DF.** Expects crops at `faces_celebdf/` and `index/celebdf_v2.csv` (same crop recipe as M8a/M9a). If the index is missing, section 4 prints what to prepare and skips scoring. Video root for raw files (if you build the index elsewhere): `~/celebdf_v2`.

**Interleaved order.** Batches stay **pure** (one generator, so GBLA’s δ_g is well-defined). Instead of 225 steps of DF then F2F then FS then NT, the sampler cycles generators every batch: DF, F2F, FS, NT, DF, … Same images per epoch as M4.

| Section | Content | Run time |
|---|---|---|
| 1 | setup, detector, load the 9 checkpoints | ~2 min |
| 2 | Priority 1 — metrics + ROC from saved preds | minutes, mostly CPU |
| 3 | Priority 5 — latency / FLOPs / memory | minutes |
| 4 | Priority 2 — Celeb-DF v2 | ~20–40 min if crops exist |
| 5 | Priorities 3–4 — multi-seed + interleaved training | ~8–11 h total, resumable |
| 6 | Combined summary + send back | seconds |

## 1. Setup

### 1.1 Settings

In [ ]:
# 1.1 Settings
import os
from pathlib import Path

DATA_ROOT = Path("/nlsasfs/home/isea/isea64/GBLA-Deepfake Detection")
RUNS_ROOT = DATA_ROOT / "runs"
FACES_FF = DATA_ROOT / "faces"
FACES_DFD = DATA_ROOT / "faces_dfd"
FACES_UADFV = DATA_ROOT / "faces_uadfv"
FACES_CELEB = DATA_ROOT / "faces_celebdf"
FF_INDEX = DATA_ROOT / "index" / "ffpp_c23.csv"
DFD_INDEX = DATA_ROOT / "index" / "dfd_c23.csv"
UADFV_INDEX = DATA_ROOT / "index" / "uadfv.csv"
CELEB_INDEX = DATA_ROOT / "index" / "celebdf_v2.csv"
CELEB_VIDEO_ROOT = Path("/nlsasfs/home/isea/isea64/celebdf_v2")   # raw videos if you build crops elsewhere
M1_DIR0 = RUNS_ROOT / "m1"
M4_DIR = RUNS_ROOT / "m4"
M6_MERGES = RUNS_ROOT / "m6" / "merges"
M8B_DIR = RUNS_ROOT / "m8b"
M9B_DIR = RUNS_ROOT / "m9b"
M9C_DIR = RUNS_ROOT / "m9c"

ARCH, PRETRAINED = "ViT-L-14", "openai"
L2_NORM = True
SEED = 0
EPOCH = 10
EVAL_BATCH = 256
LOAD_THREADS = 16
THRESH = 0.5
N_LATENCY = 50
LATENCY_WARMUP = 10

# Priority 3 / 4 training
TRAIN_SEEDS = [1, 2]                 # seed 0 already done in M4
TRAIN_LAMBDAS = [float("inf"), 0.01]
RUN_INTERLEAVED = True               # seed 0, same two lambdas, interleaved sampler
RUN_MULTI_SEED = True
W_UNIFORM, W_ALIGN = 0.5, 0.1
AUGMENT = True
BATCH_SIZE = 128
EPOCHS = 10
CYCLE_EPOCHS = 10
WARMUP_EPOCHS = 1
LR_MAX, LR_MIN = 3e-4, 1e-5
VAL_FRAMES_PER_VIDEO = 8
CACHE_IN_RAM = True
GRAD_CHECKPOINTING = False

ALL_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
TRAIN_GENERATORS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
SHORT = {"original": "real", "Deepfakes": "DF", "Face2Face": "F2F", "FaceSwap": "FS",
         "NeuralTextures": "NT", "FaceShifter": "FSh"}
G = len(TRAIN_GENERATORS)

PAPER_DFD = 96.4
PAPER_UADFV = 99.2
# GenD (CLIP ViT-L/14) Celeb-DF-v2 video AUC, Table 4 mean of 5 seeds — set to the number in the PDF before citing
PAPER_CELEB = None   # e.g. 89.x ; left None so we do not invent a paper number


def m4_run(tag, seed=SEED):
    return M4_DIR / f"gbla_lam{tag}_{ARCH}_seed{seed}"


def m1_dir(seed=SEED):
    return M1_DIR0 / f"lnclip_{ARCH}_train-DF-F2F-FS-NT_seed{seed}"


MODELS = {
    "m1": ("LNCLIP-DF repro (M1)", "state", m1_dir(0) / f"epoch_{EPOCH:02d}.pt",
           m1_dir(0) / f"test_predictions_epoch{EPOCH:02d}.csv"),
    "lam_inf": ("pooled, λ=∞ (M4)", "gbla", m4_run("inf") / f"epoch_{EPOCH:02d}.pt", m4_run("inf") / "test_predictions.csv"),
    "lam_1": ("GBLA λ=1", "gbla", m4_run("1") / f"epoch_{EPOCH:02d}.pt", m4_run("1") / "test_predictions.csv"),
    "lam_0.1": ("GBLA λ=0.1", "gbla", m4_run("0.1") / f"epoch_{EPOCH:02d}.pt", m4_run("0.1") / "test_predictions.csv"),
    "lam_0.01": ("GBLA λ=0.01 (ours)", "gbla", m4_run("0.01") / f"epoch_{EPOCH:02d}.pt", m4_run("0.01") / "test_predictions.csv"),
    "lam_0.001": ("GBLA λ=0.001", "gbla", m4_run("0.001") / f"epoch_{EPOCH:02d}.pt", m4_run("0.001") / "test_predictions.csv"),
    "lam_0": ("GBLA λ=0", "gbla", m4_run("0") / f"epoch_{EPOCH:02d}.pt", m4_run("0") / "test_predictions.csv"),
    "uniform": ("uniform merge (M6)", "flat", M6_MERGES / "hold-FSh_uniform.pt", None),
    "fisher": ("Fisher merge (M6)", "flat", M6_MERGES / "hold-FSh_fisher.pt", None),
}
MAIN = ["m1", "lam_inf", "lam_0.01"]
OURS = "lam_0.01"
M6_FSH = {"uniform": 81.43, "fisher": 88.99}
M1_FSH = 88.19

M9C_DIR.mkdir(parents=True, exist_ok=True)
missing = [k for k, (_, _, path, _) in MODELS.items() if not path.exists()]
for k in missing:
    print(f"checkpoint missing, skipped: {k} -> {MODELS[k][2]}")
MODELS = {k: v for k, v in MODELS.items() if k not in missing}
assert all(k in MODELS for k in MAIN), "a main model is missing"
print("models:", list(MODELS))
print("Celeb-DF index:", "FOUND" if CELEB_INDEX.exists() else f"MISSING -> {CELEB_INDEX}")
print("results ->", M9C_DIR)

In [ ]:
# 1.2 Imports
import json, math, random, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "M9c needs the GPU for sections 3–5 (section 2 metrics are CPU)"
DEVICE = "cuda"
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
display(pd.Series({"torch": torch.__version__, "open_clip": open_clip.__version__,
                   "GPU": torch.cuda.get_device_name(0)}, name="value").to_frame())

In [ ]:
# 1.3 Detector, θ loaders, predict / AUC helpers
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        encoder = open_clip.create_model(ARCH, pretrained=PRETRAINED, force_quick_gelu=True).visual
        encoder.proj = None
        self.encoder = encoder
        self.head = nn.Linear(encoder.transformer.width, 2)

    def forward(self, x):
        feats = self.encoder(x).float()
        f = F.normalize(feats, dim=-1)
        return self.head(f if L2_NORM else feats), f


torch.manual_seed(SEED)
model = Detector().to(DEVICE).eval()
model.requires_grad_(False)
for m in model.encoder.modules():
    if isinstance(m, nn.LayerNorm):
        m.requires_grad_(True)
model.head.requires_grad_(True)
PARAMS = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
PARAM_NAMES = [n for n, _ in PARAMS]
P = sum(p.numel() for _, p in PARAMS)
TOTAL = sum(p.numel() for p in model.parameters())
param_report = {"total": TOTAL, "trainable (deployed)": P, "frozen": TOTAL - P,
                "trainable %": round(100 * P / TOTAL, 4)}
display(pd.Series(param_report, name="parameters").to_frame())
assert P == 104_450, P


@torch.no_grad()
def set_flat(theta):
    theta = theta.to(DEVICE).float()
    assert theta.numel() == P
    offset = 0
    for _, p in PARAMS:
        n = p.numel()
        p.copy_(theta[offset:offset + n].view_as(p).to(p.dtype))
        offset += n


def load(path):
    return torch.load(path, map_location="cpu", weights_only=False)


def load_theta(kind, path):
    obj = load(path)
    if kind == "state":
        state = obj.get("trainable", obj)
        return torch.cat([state[n].reshape(-1).float() for n in PARAM_NAMES])
    if kind == "gbla":
        return obj.get("gbla", obj)["theta_s"].float()
    if kind == "flat":
        return obj["theta"].float()
    raise ValueError(kind)


THETAS = {k: load_theta(kind, path) for k, (_, kind, path, _) in MODELS.items()}
MEAN_GPU = torch.tensor(CLIP_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_GPU = torch.tensor(CLIP_STD, device=DEVICE).view(1, 3, 1, 1)


def read_rgb(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"))


def load_images(paths, desc):
    images = np.empty((len(paths), 224, 224, 3), dtype=np.uint8)

    def fill(i):
        images[i] = read_rgb(paths[i])

    with ThreadPoolExecutor(LOAD_THREADS) as pool:
        list(tqdm(pool.map(fill, range(len(paths)), chunksize=64), total=len(paths), desc=desc))
    return images


@torch.no_grad()
def predict(theta, images, desc="predict"):
    set_flat(theta)
    model.eval()
    probs = []
    for s in tqdm(range(0, len(images), EVAL_BATCH), desc=desc, leave=False):
        x = torch.from_numpy(images[s:s + EVAL_BATCH]).to(DEVICE, non_blocking=True)
        x = (x.permute(0, 3, 1, 2).float() / 255 - MEAN_GPU) / STD_GPU
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(x)
        probs.append(logits.float().softmax(1)[:, 1].cpu())
    return torch.cat(probs).numpy()


def auc(labels, scores):
    labels = np.asarray(labels)
    ranks = pd.Series(np.asarray(scores)).rank().values
    n_pos = labels.sum(); n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def average_precision(labels, scores):
    order = np.argsort(-np.asarray(scores), kind="stable")
    y = np.asarray(labels)[order]
    return float((np.cumsum(y) / np.arange(1, len(y) + 1) * y).sum() / y.sum())


def roc_points(labels, scores):
    order = np.argsort(-np.asarray(scores), kind="stable")
    y = np.asarray(labels)[order]
    tpr = np.concatenate([[0], np.cumsum(y) / max(y.sum(), 1)])
    fpr = np.concatenate([[0], np.cumsum(1 - y) / max((1 - y).sum(), 1)])
    return fpr, tpr


def eer(labels, scores):
    fpr, tpr = roc_points(labels, scores)
    i = int(np.argmin(np.abs((1 - tpr) - fpr)))
    return float((fpr[i] + 1 - tpr[i]) / 2)


def classification_at_threshold(labels, scores, thr=THRESH):
    y = np.asarray(labels).astype(int)
    pred = (np.asarray(scores) >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    spec = tn / (tn + fp) if tn + fp else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else float("nan")
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "precision": 100 * prec, "recall": 100 * rec,
            "specificity": 100 * spec, "f1": 100 * f1, "accuracy": 100 * (tp + tn) / max(len(y), 1)}


print({k: f"{v.numel():,}" for k, v in THETAS.items()})

## 2. Priority 1 — Classification metrics from saved predictions (CPU)

Uses already-written prediction CSVs. No re-inference.

| set | file |
|---|---|
| FF++ test | each model’s `test_predictions.csv` (M1 / M4); merges use FaceShifter check only in M6 numbers |
| DFD | `runs/m8b/pred_<key>.csv` |
| UADFV | `runs/m9b/pred_<key>.csv` |

Threshold = 0.5 on the **video** score (mean frame probability), same operating point as the acc columns in M8b/M9b. Confusion counts are TP/TN/FP/FN at that threshold. Also: precision, recall (fake = positive), specificity, F1, video AUC, and the UADFV ROC plot.

In [ ]:
# 2.1 Metrics from saved predictions
SEEN = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]


def video_table_from_frame_df(df, label_col="label", score_col="prob"):
    """Aggregate frame rows to one score per video_id (and generator if present)."""
    keys = ["video_id"] + (["generator"] if "generator" in df.columns else [])
    return df.groupby(keys, as_index=False).agg(label=(label_col, "first"), score=(score_col, "mean"))


def metrics_block(name, vid):
    y, s = vid.label.values, vid.score.values
    row = {"set": name, "n_videos": len(vid), "video AUC": 100 * auc(y, s),
           "video AP": 100 * average_precision(y, s), "video EER": 100 * eer(y, s)}
    row.update(classification_at_threshold(y, s, THRESH))
    return row


rows, confusion_rows = [], []

# FF++ test from M4/M1 CSVs
for k, (name, _, _, pred_path) in MODELS.items():
    if pred_path is None or not Path(pred_path).exists():
        print(f"FF++ preds missing for {k}, skip")
        continue
    pred = pd.read_csv(pred_path)
    if "prob" not in pred.columns and "score" in pred.columns:
        pred = pred.rename(columns={"score": "prob"})
    # seen combined: each generator vs the same real test videos
    if "generator" in pred.columns:
        for tag, gens in [("FF++ seen", SEEN), ("FF++ FaceShifter", ["FaceShifter"])]:
            sub = pred[pred.generator.isin(["original", *gens])]
            vid = video_table_from_frame_df(sub)
            r = metrics_block(f"{tag} | {name}", vid)
            r["model_key"] = k
            rows.append(r)
            confusion_rows.append({"set": tag, "model": name, "key": k,
                                   **{c: r[c] for c in ("tp", "tn", "fp", "fn", "precision", "recall", "specificity", "f1")}})
    else:
        vid = video_table_from_frame_df(pred)
        r = metrics_block(f"FF++ | {name}", vid); r["model_key"] = k; rows.append(r)

# DFD / UADFV from M8b / M9b pred files
for set_name, index_path, pred_dir, faces_note in [
        ("DFD", DFD_INDEX, M8B_DIR, "m8b"),
        ("UADFV", UADFV_INDEX, M9B_DIR, "m9b")]:
    if not index_path.exists():
        print(f"{set_name}: index missing, skip"); continue
    index = pd.read_csv(index_path, dtype={"video_id": str})
    for k, (name, *_ ) in MODELS.items():
        path = pred_dir / f"pred_{k}.csv"
        if not path.exists():
            print(f"{set_name} pred missing: {path.name}"); continue
        saved = pd.read_csv(path)
        if len(saved) != len(index):
            print(f"{set_name} {k}: length mismatch, skip"); continue
        df = index.copy()
        df["prob"] = saved.prob.values
        vid = video_table_from_frame_df(df)
        r = metrics_block(f"{set_name} | {name}", vid); r["model_key"] = k; rows.append(r)
        confusion_rows.append({"set": set_name, "model": name, "key": k,
                               **{c: r[c] for c in ("tp", "tn", "fp", "fn", "precision", "recall", "specificity", "f1")}})

metrics_table = pd.DataFrame(rows)
confusion_table = pd.DataFrame(confusion_rows)
display(metrics_table.round(2))
display(confusion_table.round(2))
metrics_table.to_csv(M9C_DIR / "classification_metrics.csv", index=False)
confusion_table.to_csv(M9C_DIR / "confusion_counts.csv", index=False)

# UADFV ROC for main models (+ paper note)
if UADFV_INDEX.exists():
    uidx = pd.read_csv(UADFV_INDEX, dtype={"video_id": str, "pair_id": str})
    uvid = uidx.groupby("video_id", as_index=False).agg(label=("label", "first"))
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for k in [k for k in MAIN + ["fisher", "uniform"] if k in MODELS]:
        path = M9B_DIR / f"pred_{k}.csv"
        if not path.exists():
            continue
        probs = pd.read_csv(path).prob.values
        scores = uidx.assign(prob=probs).groupby("video_id").prob.mean().reindex(uvid.video_id).values
        fpr, tpr = roc_points(uvid.label.values, scores)
        ax.plot(fpr, tpr, lw=1.5, label=f"{MODELS[k][0]} ({100 * auc(uvid.label.values, scores):.2f})")
    ax.plot([0, 1], [0, 1], "k:", lw=0.8)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("UADFV video ROC (threshold-free)")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(M9C_DIR / "uadfv_roc.png", dpi=150); plt.show()

## 3. Priority 5 — Inference latency, FLOPs, memory

Measured on one A100 with the deployed model (θ_s of GBLA λ=0.01). Batch = 1 and batch = 32 face crops (224²). FLOPs via a backward-free fvcore/thop count if installed, otherwise the closed-form ViT-L/14 estimate used in the literature (~61.6 GFLOPs / image at 224 for ViT-L/14 without the text tower). Peak allocated CUDA memory during a forward pass.

In [ ]:
# 3. Latency / FLOPs / memory
set_flat(THETAS[OURS])
model.eval()
WIDTH = model.encoder.transformer.width


def make_batch(bs):
    x = torch.randn(bs, 3, 224, 224, device=DEVICE)
    return (x - MEAN_GPU) / STD_GPU


def time_forward(bs, n=N_LATENCY, warmup=LATENCY_WARMUP):
    x = make_batch(bs)
    for _ in range(warmup):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            model(x)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            model(x)
    torch.cuda.synchronize()
    ms = (time.time() - t0) * 1000 / n
    return ms, ms / bs


# FLOPs
flops_src, flops_img = "estimate", None
try:
    from fvcore.nn import FlopCountAnalysis
    flops_img = FlopCountAnalysis(model, make_batch(1)).total() / 1e9
    flops_src = "fvcore"
except Exception:
    try:
        from thop import profile
        flops, _ = profile(model, inputs=(make_batch(1),), verbose=False)
        flops_img = flops / 1e9
        flops_src = "thop"
    except Exception:
        # ViT-L/14 @224: 24 blocks, width 1024, MLP 4096, 16 heads — ~61.6 GFLOPs (Dosovitskiy et al. / open_clip scale)
        flops_img = 61.6
        flops_src = "closed-form ViT-L/14 @224 (no text tower; LayerNorm/head negligible)"

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
with torch.autocast("cuda", dtype=torch.bfloat16):
    model(make_batch(32))
torch.cuda.synchronize()
peak_mb = torch.cuda.max_memory_allocated() / 1e6

lat_rows = []
for bs in (1, 32):
    ms_batch, ms_img = time_forward(bs)
    lat_rows.append({"batch size": bs, "ms / batch": round(ms_batch, 2), "ms / image": round(ms_img, 3),
                     "images / s": round(1000 / ms_img, 1)})
latency = pd.DataFrame(lat_rows)
display(latency)
cost = pd.Series({
    "trainable parameters": P,
    "total parameters": TOTAL,
    "GFLOPs / image": round(flops_img, 2),
    "FLOPs source": flops_src,
    "peak CUDA allocated (MB), batch 32": round(peak_mb, 1),
    "GPU": torch.cuda.get_device_name(0),
}, name="value").to_frame()
display(cost)
latency.to_csv(M9C_DIR / "latency.csv", index=False)
cost.to_csv(M9C_DIR / "compute_cost.csv")
(M9C_DIR / "compute_cost.json").write_text(json.dumps({
    "latency": lat_rows, "gflops_per_image": flops_img, "flops_source": flops_src,
    "peak_cuda_mb_batch32": peak_mb, "trainable": P, "total": TOTAL,
}, indent=2))

## 4. Priority 2 — Celeb-DF v2 (9 checkpoints, no training)

**Protocol.** Same as DFD/UADFV: video score = mean of 32 frame probabilities; metric = video AUC. Paper reference: GenD CLIP ViT-L/14 Celeb-DF-v2 ≈ **89.4** (Table 4, mean of 5 seeds; confirm against the PDF if you cite it).

**Data.** Crops must already exist at `faces_celebdf/` with `index/celebdf_v2.csv` (columns at least: `label`, `video_id`, `path`, `frame_idx`), built with the same MTCNN / 1.3× / 224 pipeline as M8a. If the index is missing, this section stops after printing the expected paths — build the crops first (same style as M8a; raw videos in `celebdf_v2/`).

In [ ]:
# 4. Celeb-DF v2 evaluation
celeb_summary = {"status": "skipped", "reason": None}
if not CELEB_INDEX.exists():
    celeb_summary["reason"] = f"missing {CELEB_INDEX}; prepare faces_celebdf + index like M8a"
    print("SKIP Celeb-DF:", celeb_summary["reason"])
    print("raw videos expected at:", CELEB_VIDEO_ROOT, "(exists)" if CELEB_VIDEO_ROOT.exists() else "(missing)")
else:
    celeb = pd.read_csv(CELEB_INDEX, dtype={"video_id": str})
    assert {"label", "video_id", "path"}.issubset(celeb.columns), celeb.columns.tolist()
    cvid = celeb.groupby("video_id", as_index=False).agg(label=("label", "first"), crops=("path", "size"))
    print(f"Celeb-DF crops: {len(celeb):,}   videos real={(cvid.label == 0).sum()} fake={(cvid.label == 1).sum()}")
    CELEB_IMAGES = load_images([str(FACES_CELEB / p) for p in celeb.path], desc="Celeb-DF crops")
    celeb_rows = []
    for k, (name, *_ ) in MODELS.items():
        path = M9C_DIR / f"pred_celeb_{k}.csv"
        if path.exists() and len(pd.read_csv(path)) == len(celeb):
            probs = pd.read_csv(path).prob.values
            print(f"{k}: reused {path.name}")
        else:
            t0 = time.time()
            probs = predict(THETAS[k], CELEB_IMAGES, desc=f"celeb {k}")
            pd.DataFrame({"video_id": celeb.video_id, "path": celeb.path, "prob": probs}).to_csv(path, index=False)
            print(f"{k}: {time.time() - t0:.0f} s")
        scores = celeb.assign(prob=probs).groupby("video_id").prob.mean().reindex(cvid.video_id).values
        y = cvid.label.values
        row = {"model": name, "key": k, "video AUC": 100 * auc(y, scores),
               "video AP": 100 * average_precision(y, scores), "video EER": 100 * eer(y, scores)}
        row.update(classification_at_threshold(y, scores))
        celeb_rows.append(row)
    if PAPER_CELEB is not None:
        celeb_rows.append({"model": "GenD (CLIP) paper, 5 seeds", "key": "paper", "video AUC": PAPER_CELEB})
    else:
        print("PAPER_CELEB is None — fill GenD Table 4 Celeb-DF number in settings before citing")
    celeb_table = pd.DataFrame(celeb_rows).set_index("model")
    display(celeb_table.round(2))
    celeb_table.to_csv(M9C_DIR / "celebdf_main_table.csv")
    celeb_summary = {"status": "ok", "table": celeb_table.round(3).reset_index().astype(str).to_dict("records")}
    del CELEB_IMAGES

## 5. Priorities 3–4 — Multi-seed and interleaved training

Same recipe as M4: FF++ train crops, round-robin real ownership, Adam, 10 epochs, deploy θ_s only.

| run folder pattern | meaning |
|---|---|
| `runs/m4/gbla_lam{inf\|0.01}_ViT-L-14_seed{1\|2}/` | Priority 3: new seeds, **block** order (as M4) |
| `runs/m9c/gbla_interleaved_lam{inf\|0.01}_ViT-L-14_seed0/` | Priority 4: seed 0, **interleaved** pure batches |

Seed 0 block-order runs already exist in M4 and are **not** retrained. After each new run finishes, FF++ test video AUC is written to its `results.json`. Section 5.3 gathers seed-0 (M4) + new seeds + interleaved into one table.

In [ ]:
# 5.1 Training data + loaders (shared by all Priority 3/4 runs)
import torchvision
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import tv_tensors
from torchvision.io import ImageReadMode, read_image
from torchvision.transforms import v2 as T

INDEX_CSV = FF_INDEX
FACES_ROOT = FACES_FF


def usable_cpus():
    n = len(os.sched_getaffinity(0))
    for quota_file, period_file in [("/sys/fs/cgroup/cpu.max", None),
                                    ("/sys/fs/cgroup/cpu/cpu.cfs_quota_us", "/sys/fs/cgroup/cpu/cpu.cfs_period_us")]:
        try:
            if period_file is None:
                quota, period = Path(quota_file).read_text().split()
            else:
                quota, period = Path(quota_file).read_text().strip(), Path(period_file).read_text().strip()
            if quota not in ("max", "-1"):
                n = min(n, max(1, int(quota) // int(period)))
        except (OSError, ValueError):
            pass
    return n


def available_memory_gb():
    candidates = []
    try:
        meminfo = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
        candidates.append(int(meminfo["MemAvailable"].split()[0]) * 1024)
    except (OSError, KeyError, ValueError):
        pass
    for limit_file, usage_file, stat_file, cache_key in [
            ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.stat", "file"),
            ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes",
             "/sys/fs/cgroup/memory/memory.stat", "cache")]:
        try:
            limit = Path(limit_file).read_text().strip()
            if not limit.isdigit() or int(limit) >= 1 << 60:
                continue
            usage = int(Path(usage_file).read_text().strip())
            stat = dict(line.split() for line in Path(stat_file).read_text().splitlines())
            candidates.append(int(limit) - usage + int(stat.get(cache_key, 0)))
        except (OSError, ValueError):
            pass
    return min(candidates) / 1e9 if candidates else 0.0


CPUS = usable_cpus()
NUM_WORKERS = max(1, min(16, CPUS - 1))
torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

index = pd.read_csv(INDEX_CSV, dtype={"video_id": str, "target_id": str, "source_id": str}, keep_default_na=False)
CATEGORIES = ["original"] + ALL_GENERATORS
GEN_CODE = {g: k for k, g in enumerate(CATEGORIES)}


def subset(split, generators):
    keep = (index.split == split) & index.generator.isin(["original", *generators])
    return index[keep].reset_index(drop=True)


train_df = subset("train", TRAIN_GENERATORS)
val_all = subset("val", TRAIN_GENERATORS)
frames_per_video = int(val_all.groupby(["generator", "video_id"]).size().max())
keep_every = max(1, frames_per_video // VAL_FRAMES_PER_VIDEO)
frame_rank = val_all.groupby(["generator", "video_id"]).frame_idx.rank(method="first").astype(int) - 1
val_df = val_all[frame_rank % keep_every == 0].reset_index(drop=True)
test_df = subset("test", ALL_GENERATORS)


def load_images_df(df, desc):
    paths = [FACES_ROOT / p for p in df.path]
    first = np.asarray(Image.open(paths[0]).convert("RGB"))
    out = np.empty((len(paths), *first.shape), dtype=np.uint8)

    def work(k):
        out[k] = np.asarray(Image.open(paths[k]).convert("RGB"))

    with ThreadPoolExecutor(max_workers=min(32, 4 * CPUS)) as pool:
        list(tqdm(pool.map(work, range(len(paths))), total=len(paths), desc=desc, mininterval=5))
    return out


images = {}
budget = 0.6 * available_memory_gb()
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    need = len(df) * 224 * 224 * 3 / 1e9
    if CACHE_IN_RAM and need <= budget:
        t = time.time()
        images[name] = load_images_df(df, f"{name} crops -> RAM")
        budget -= need
        print(f"{name}: {len(df):,} crops, {need:.1f} GB in {(time.time() - t) / 60:.1f} min")
    else:
        images[name] = None
        print(f"{name}: from disk (needs {need:.1f} GB)")


def make_transform(train):
    steps = []
    if train and AUGMENT:
        steps += [T.RandomHorizontalFlip(0.5),
                  T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
                  T.RandomApply([T.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))], p=0.1),
                  T.ColorJitter(brightness=0.1, contrast=0.1),
                  T.JPEG(quality=(40, 100))]
    steps += [T.ToDtype(torch.float32, scale=True), T.Normalize(CLIP_MEAN, CLIP_STD)]
    return T.Compose(steps)


class FaceDataset(Dataset):
    def __init__(self, df, train, images=None):
        self.paths = [str(FACES_ROOT / p) for p in df.path]
        self.images = images
        self.labels = torch.tensor(df.label.values, dtype=torch.long)
        self.gens = torch.tensor(df.generator.map(GEN_CODE).values, dtype=torch.long)
        self.transform = make_transform(train)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        if self.images is not None:
            image = torch.from_numpy(np.array(self.images[i])).permute(2, 0, 1)
        else:
            image = read_image(self.paths[i], mode=ImageReadMode.RGB)
        return self.transform(tv_tensors.Image(image)).as_subclass(torch.Tensor), self.labels[i], self.gens[i]


class BlockSampler(Sampler):
    """M4 order: one contiguous block per generator per epoch."""

    def __init__(self, df, seed):
        self.seed, self.epoch = seed, 0
        gen = df.generator.values
        real_idx = np.flatnonzero(gen == "original")
        self.blocks = {}
        for k, g in enumerate(TRAIN_GENERATORS):
            fakes = np.flatnonzero(gen == g)
            reals = real_idx[k::G][: len(fakes) // G]
            self.blocks[g] = (fakes, reals)
        self.n_per = len(self.blocks[TRAIN_GENERATORS[0]][0]) + len(self.blocks[TRAIN_GENERATORS[0]][1])

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        order_rng = np.random.default_rng([self.seed, self.epoch, 1])
        gens = list(TRAIN_GENERATORS)
        order_rng.shuffle(gens)
        out = []
        for g in gens:
            fakes, reals = self.blocks[g]
            rng = np.random.default_rng([self.seed, ALL_GENERATORS.index(g), self.epoch])
            out.extend(rng.permutation(np.concatenate([fakes, reals])).tolist())
        return iter(out)

    def __len__(self):
        return self.n_per * G


class InterleavedSampler(Sampler):
    """Same ownership as BlockSampler, but generators cycle every batch (pure batches)."""

    def __init__(self, df, seed, batch_size=BATCH_SIZE):
        self.seed, self.epoch, self.batch_size = seed, 0, batch_size
        gen = df.generator.values
        real_idx = np.flatnonzero(gen == "original")
        self.blocks = {}
        for k, g in enumerate(TRAIN_GENERATORS):
            fakes = np.flatnonzero(gen == g)
            reals = real_idx[k::G][: len(fakes) // G]
            self.blocks[g] = (fakes, reals)
        self.n_per = len(self.blocks[TRAIN_GENERATORS[0]][0]) + len(self.blocks[TRAIN_GENERATORS[0]][1])
        assert self.n_per % batch_size == 0, (self.n_per, batch_size)

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        perms = {}
        for g in TRAIN_GENERATORS:
            fakes, reals = self.blocks[g]
            rng = np.random.default_rng([self.seed, ALL_GENERATORS.index(g), self.epoch, 7])
            perms[g] = rng.permutation(np.concatenate([fakes, reals]))
        n_batches_per = self.n_per // self.batch_size
        order_rng = np.random.default_rng([self.seed, self.epoch, 3])
        # list of (generator, batch_index) then shuffle while keeping pure batches
        schedule = [(g, b) for b in range(n_batches_per) for g in TRAIN_GENERATORS]
        order_rng.shuffle(schedule)
        out = []
        for g, b in schedule:
            s = b * self.batch_size
            out.extend(perms[g][s:s + self.batch_size].tolist())
        return iter(out)

    def __len__(self):
        return self.n_per * G


train_ds = FaceDataset(train_df, train=True, images=images["train"])
print("train crops:", len(train_ds), " workers:", NUM_WORKERS)

In [ ]:
# 5.2 GBLA training core (seed + sampler mode)
def get_flat():
    return torch.cat([p.detach().reshape(-1) for _, p in PARAMS]).float()


@torch.no_grad()
def set_flat_train(theta):
    offset = 0
    for _, p in PARAMS:
        n = p.numel()
        p.copy_(theta[offset:offset + n].view_as(p))
        offset += n


def flat_grad():
    return torch.cat([p.grad.reshape(-1) for _, p in PARAMS]).float()


THETA0 = get_flat().clone()


def compute_loss(logits, f, y):
    ce_each = F.cross_entropy(logits, y, reduction="none")
    ce = ce_each.mean()
    n = len(y)
    i, j = torch.triu_indices(n, n, offset=1, device=f.device)
    d2 = (2 - 2 * (f @ f.T)[i, j]).clamp(min=0)
    same = y[i] == y[j]
    align = d2[same].mean() if same.any() else d2.new_zeros(())
    uniform = torch.exp(-2 * d2).mean().clamp(min=1e-6).log()
    loss = ce + W_ALIGN * align + W_UNIFORM * uniform
    parts = {"loss": loss.item(), "ce": ce.item(), "align": align.item(), "uniform": uniform.item()}
    return loss, parts, ce_each.detach()


def lr_at(step, steps_per_epoch):
    cycle = CYCLE_EPOCHS * steps_per_epoch
    warm = WARMUP_EPOCHS * steps_per_epoch
    pos = step % cycle
    if pos < warm:
        return LR_MIN + (LR_MAX - LR_MIN) * pos / warm
    frac = (pos - warm) / max(cycle - warm, 1)
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + math.cos(math.pi * frac))


class GBLAState:
    def __init__(self):
        self.theta_s = THETA0.detach().clone()
        self.delta = {g: torch.zeros(P, device=DEVICE) for g in TRAIN_GENERATORS}

    def theta(self, g):
        return self.theta_s + self.delta[g]

    def save(self):
        return {"theta_s": self.theta_s.detach().cpu(),
                "delta": {g: v.detach().cpu() for g, v in self.delta.items()}}

    def load(self, obj):
        self.theta_s = obj["theta_s"].to(DEVICE).float()
        self.delta = {g: obj["delta"][g].to(DEVICE).float() for g in TRAIN_GENERATORS}


def adam_step(param, grad, m, v, t, lr):
    b1, b2, eps = 0.9, 0.999, 1e-8
    t += 1
    m.mul_(b1).add_(grad, alpha=1 - b1)
    v.mul_(b2).addcmul_(grad, grad, value=1 - b2)
    param.addcdiv_(m / (1 - b1 ** t), (v / (1 - b2 ** t)).sqrt().add_(eps), value=-lr)


class AdamSlots:
    def __init__(self):
        self.m_s = torch.zeros(P, device=DEVICE); self.v_s = torch.zeros(P, device=DEVICE)
        self.t_s = torch.zeros((), dtype=torch.long, device=DEVICE)
        self.m_d = {g: torch.zeros(P, device=DEVICE) for g in TRAIN_GENERATORS}
        self.v_d = {g: torch.zeros(P, device=DEVICE) for g in TRAIN_GENERATORS}
        self.t_d = {g: torch.zeros((), dtype=torch.long, device=DEVICE) for g in TRAIN_GENERATORS}

    def state_dict(self):
        return {"m_s": self.m_s.cpu(), "v_s": self.v_s.cpu(), "t_s": int(self.t_s),
                "m_d": {g: v.cpu() for g, v in self.m_d.items()},
                "v_d": {g: v.cpu() for g, v in self.v_d.items()},
                "t_d": {g: int(v) for g, v in self.t_d.items()}}

    def load_state_dict(self, s):
        self.m_s.copy_(s["m_s"].to(DEVICE)); self.v_s.copy_(s["v_s"].to(DEVICE)); self.t_s.fill_(s["t_s"])
        for g in TRAIN_GENERATORS:
            self.m_d[g].copy_(s["m_d"][g].to(DEVICE)); self.v_d[g].copy_(s["v_d"][g].to(DEVICE))
            self.t_d[g].fill_(s["t_d"][g])


def train_step_gbla(state, opt, x, y, g, lr, lam):
    set_flat_train(state.theta(g))
    model.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        logits, f = model(x)
    loss, parts, ce_each = compute_loss(logits.float(), f.float(), y)
    if lam < float("inf"):
        penalty = lam * state.delta[g].detach().pow(2).sum()
        parts["penalty"] = float(penalty); parts["loss"] += parts["penalty"]
    else:
        parts["penalty"] = 0.0
    loss.backward()
    grad = flat_grad().detach()
    adam_step(state.theta_s, grad, opt.m_s, opt.v_s, opt.t_s, lr)
    if lam < float("inf"):
        adam_step(state.delta[g], grad + (2 * lam) * state.delta[g], opt.m_d[g], opt.v_d[g], opt.t_d[g], lr)
    return parts, ce_each


def to_model_input(uint8_batch):
    x = torch.as_tensor(uint8_batch).to(DEVICE, non_blocking=True)
    return (x.permute(0, 3, 1, 2).float() / 255 - MEAN_GPU) / STD_GPU


@torch.no_grad()
def predict_df(theta, df, imgs=None, desc="predict"):
    set_flat_train(theta.detach()); model.eval()
    probs = []
    if imgs is not None:
        for s in tqdm(range(0, len(df), EVAL_BATCH), desc=desc, leave=False):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits, _ = model(to_model_input(imgs[s:s + EVAL_BATCH]))
            probs.append(logits.float().softmax(1)[:, 1].cpu())
    else:
        loader = DataLoader(FaceDataset(df, train=False), batch_size=EVAL_BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        for x, _, _ in tqdm(loader, desc=desc, leave=False):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                logits, _ = model(x.to(DEVICE, non_blocking=True))
            probs.append(logits.float().softmax(1)[:, 1].cpu())
    model.train()
    out = df[["split", "label", "generator", "video_id", "frame_idx", "path"]].copy()
    out["prob"] = torch.cat(probs).numpy()
    return out


def metrics_table(pred):
    groups = {"seen combined (" + "+".join(SHORT[g] for g in TRAIN_GENERATORS) + ")": TRAIN_GENERATORS}
    groups.update({g: [g] for g in ALL_GENERATORS})
    rows = []
    for name, gens in groups.items():
        sub = pred[pred.generator.isin(["original", *gens])]
        if (sub.label == 1).sum() == 0:
            continue
        vid = sub.groupby(["generator", "video_id"], as_index=False).agg(label=("label", "first"), prob=("prob", "mean"))
        rows.append({"test set": name,
                     "status": "seen" if set(gens) <= set(TRAIN_GENERATORS) else "UNSEEN",
                     "frame AUC": 100 * auc(sub.label, sub.prob),
                     "video AUC": 100 * auc(vid.label, vid.prob),
                     "video acc @0.5": 100 * float(((vid.prob > 0.5) == vid.label).mean())})
    return pd.DataFrame(rows).round(2)


def save_atomic(obj, path):
    tmp = path.with_name(path.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, path)


def lam_tag(lam):
    return "inf" if lam == float("inf") else f"{lam:g}"


def run_dir_for(lam, seed, order="block"):
    if order == "block":
        return M4_DIR / f"gbla_lam{lam_tag(lam)}_{ARCH}_seed{seed}"
    return M9C_DIR / f"gbla_interleaved_lam{lam_tag(lam)}_{ARCH}_seed{seed}"


def check_config(run_dir, config):
    path = run_dir / "config.json"
    dumped = json.loads(json.dumps(config))
    if path.exists():
        assert json.loads(path.read_text()) == dumped, f"{run_dir} settings differ; delete the folder to restart"
    else:
        path.write_text(json.dumps(dumped, indent=1))


COMMON_TRAIN = {
    "arch": ARCH, "pretrained": PRETRAINED, "quick_gelu": True, "l2_norm": L2_NORM,
    "w_uniform": W_UNIFORM, "w_align": W_ALIGN, "augment": AUGMENT,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "cycle_epochs": CYCLE_EPOCHS,
    "warmup_epochs": WARMUP_EPOCHS, "lr_max": LR_MAX, "lr_min": LR_MIN,
    "method": "GBLA: theta_g = theta_s + delta_g, penalty lambda * ||delta_g||^2",
    "reals": "round-robin, fixed for the run, 1/G per generator",
    "deployed": "theta_s only",
}

In [ ]:
# 5.3 Train runs (resumable): multi-seed + interleaved
def train_one(lam, seed, order="block"):
    run_dir = run_dir_for(lam, seed, order)
    run_dir.mkdir(parents=True, exist_ok=True)
    check_config(run_dir, {**COMMON_TRAIN, "lambda": None if lam == float("inf") else lam,
                           "lambda_tag": lam_tag(lam), "seed": seed, "batch_order": order})
    # rebuild initialisation for this seed
    global THETA0
    torch.manual_seed(seed)
    # reset trainable tensors to a fresh pretrained+head init
    fresh = Detector().to(DEVICE)
    fresh.requires_grad_(False)
    for m in fresh.encoder.modules():
        if isinstance(m, nn.LayerNorm):
            m.requires_grad_(True)
    fresh.head.requires_grad_(True)
    with torch.no_grad():
        for (n, p), (_, q) in zip(PARAMS, [(n, p) for n, p in fresh.named_parameters() if p.requires_grad]):
            p.copy_(q)
    del fresh
    THETA0 = get_flat().clone()

    SamplerCls = BlockSampler if order == "block" else InterleavedSampler
    sampler = SamplerCls(train_df, seed)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    assert len(loader) == 900, len(loader)

    state, opt = GBLAState(), AdamSlots()
    history, start_epoch = [], 0
    if (run_dir / "last.pt").exists():
        ckpt = load(run_dir / "last.pt")
        state.load(ckpt["gbla"]); opt.load_state_dict(ckpt["adam"])
        history, start_epoch = ckpt["history"], ckpt["epoch"]
    print(f"\n===== lam={lam_tag(lam)} seed={seed} order={order}: {start_epoch}/{EPOCHS} =====")

    steps_per_epoch = len(loader)
    for epoch in range(start_epoch, EPOCHS):
        sampler.set_epoch(epoch)
        model.train()
        sums, n_steps, t0 = defaultdict(float), 0, time.time()
        ce_sum = torch.zeros(len(CATEGORIES), device=DEVICE)
        ce_cnt = torch.zeros(len(CATEGORIES), device=DEVICE)
        for i, (x, y, gen) in enumerate(tqdm(loader, desc=f"{lam_tag(lam)} s{seed} {order} ep{epoch+1}", leave=False)):
            x, y, gen = (t.to(DEVICE, non_blocking=True) for t in (x, y, gen))
            fake_codes = gen[gen != GEN_CODE["original"]]
            assert len(fake_codes) > 0 and torch.all(fake_codes == fake_codes[0]), "mixed-generator batch"
            g = CATEGORIES[int(fake_codes[0])]
            lr = lr_at(epoch * steps_per_epoch + i, steps_per_epoch)
            parts, ce_each = train_step_gbla(state, opt, x, y, g, lr, lam)
            for k, v in parts.items():
                sums[k] += v
            ce_sum += torch.bincount(gen, weights=ce_each, minlength=len(CATEGORIES))
            ce_cnt += torch.bincount(gen, minlength=len(CATEGORIES))
            n_steps += 1
        val_table = metrics_table(predict_df(state.theta_s, val_df, images["val"], desc="val"))
        row = {"epoch": epoch + 1, "lr (end)": float(lr)}
        row.update({k: v / n_steps for k, v in sums.items()})
        row.update({f"CE {SHORT[c]}": float(ce_sum[k] / ce_cnt[k])
                    for k, c in enumerate(CATEGORIES) if ce_cnt[k] > 0})
        row.update({f"|d| {SHORT[g]}": float(state.delta[g].norm()) for g in TRAIN_GENERATORS})
        row.update({"val frame AUC": float(val_table["frame AUC"].iloc[0]),
                    "val video AUC": float(val_table["video AUC"].iloc[0]),
                    "minutes": (time.time() - t0) / 60})
        history.append(row)
        # M4-compatible: epoch_XX.pt is the gbla dict (theta_s, delta); last.pt has adam + history
        save_atomic(state.save(), run_dir / f"epoch_{epoch + 1:02d}.pt")
        save_atomic({"epoch": epoch + 1, "gbla": state.save(), "adam": opt.state_dict(), "history": history},
                    run_dir / "last.pt")
        print(" | ".join(f"{k} {v:.4g}" if isinstance(v, float) else f"{k} {v}" for k, v in row.items()))

    if not (run_dir / "results.json").exists():
        pred = predict_df(state.theta_s, test_df, images["test"], desc="test")
        pred.to_csv(run_dir / "test_predictions.csv", index=False)
        table = metrics_table(pred)
        display(table)
        (run_dir / "results.json").write_text(json.dumps(
            {"lambda": lam_tag(lam), "seed": seed, "batch_order": order,
             "test": table.to_dict(orient="records"), "history": history,
             "delta_norms": {SHORT[g]: float(state.delta[g].norm()) for g in TRAIN_GENERATORS}}, indent=1))
    del state, opt, loader
    torch.cuda.empty_cache()


jobs = []
if RUN_MULTI_SEED:
    for seed in TRAIN_SEEDS:
        for lam in TRAIN_LAMBDAS:
            jobs.append((lam, seed, "block"))
if RUN_INTERLEAVED:
    for lam in TRAIN_LAMBDAS:
        jobs.append((lam, 0, "interleaved"))

for lam, seed, order in jobs:
    train_one(lam, seed, order)
print("Priority 3/4 training finished")

## 5.4 Gather seed / order results

FF++ test video AUC for seed 0 (M4 block order), new seeds, and interleaved seed 0.

In [ ]:
# 5.4 Seed / order table
def video_auc_from_records(records):
    t = pd.DataFrame(records).set_index("test set")
    row = {"seen": t.iloc[0]["video AUC"]}
    row.update({SHORT[g]: t.loc[g, "video AUC"] for g in ALL_GENERATORS})
    return row


seed_rows = []
# seed 0 block from M4
for lam in TRAIN_LAMBDAS:
    path = run_dir_for(lam, 0, "block") / "results.json"
    if path.exists():
        res = json.loads(path.read_text())
        seed_rows.append({"lambda": lam_tag(lam), "seed": 0, "order": "block",
                          **video_auc_from_records(res["test"])})
# new runs
for lam, seed, order in jobs:
    path = run_dir_for(lam, seed, order) / "results.json"
    if not path.exists():
        print("missing", path); continue
    res = json.loads(path.read_text())
    seed_rows.append({"lambda": lam_tag(lam), "seed": seed, "order": order,
                      **video_auc_from_records(res["test"])})

seed_table = pd.DataFrame(seed_rows)
if len(seed_table):
    display(seed_table.round(2))
    seed_table.to_csv(M9C_DIR / "seed_order_ffpp.csv", index=False)
    # mean ± std over seeds 0,1,2 for block order
    block = seed_table[seed_table.order == "block"]
    if block.seed.nunique() >= 2:
        summary = block.groupby("lambda")[["seen", "NT", "FSh"]].agg(["mean", "std", "count"])
        display(summary.round(2))
        summary.to_csv(M9C_DIR / "seed_summary_block.csv")

## 6. Combined summary

Writes `runs/m9c/summary.json` with pointers to every table produced above.

In [ ]:
# 6. Summary
summary = {
    "scope": "prior GBLA / LN-CLIP experiment only (104,450 trainable)",
    "parameters": param_report,
    "priority1_metrics_csv": str(M9C_DIR / "classification_metrics.csv"),
    "priority1_confusion_csv": str(M9C_DIR / "confusion_counts.csv"),
    "priority1_uadfv_roc": str(M9C_DIR / "uadfv_roc.png"),
    "priority5_compute": json.loads((M9C_DIR / "compute_cost.json").read_text()) if (M9C_DIR / "compute_cost.json").exists() else None,
    "priority2_celeb": celeb_summary,
    "priority3_4_seed_order_csv": str(M9C_DIR / "seed_order_ffpp.csv"),
    "models": list(MODELS),
}
(M9C_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))
print(json.dumps({k: summary[k] for k in ("parameters", "priority2_celeb")}, indent=1))

## 7. Send results back

1. Save this notebook (Ctrl+S) with outputs.
2. From the repository folder on C-DAC:

```bash
git pull
git add M9c.ipynb
git commit -m "M9c GBLA paper extras"
git push
```

When git asks for a password, paste your GitHub token. Do not put the token in this notebook or any file.

**Order.** Section 2 (metrics) can run even if later sections are pending. Section 4 skips cleanly without Celeb-DF crops. Section 5 resumes per run folder. Delete a run folder to redo that seed/order.

**What I need.** Classification + confusion tables, UADFV ROC, compute_cost.json, Celeb-DF table (or the skip reason), seed_order_ffpp.csv / seed summary, and this notebook with outputs.